# 01 — Literary Text Processing (Invisible Cities)

This notebook processes the literary text into structured units.

Steps:
1. Load raw text
2. Segment into units (city descriptions)
3. Clean and normalise text
4. Extract keywords
5. Save structured dataset

In [1]:
import re
import json
import pandas as pd
from pathlib import Path

In [2]:
BASE_DIR = Path.cwd().parent

RAW_TEXT_PATH = BASE_DIR / "data" / "raw" / "literary_text" / "invisible_cities.txt"
OUTPUT_DIR = BASE_DIR / "data" / "processed" / "text"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / "invisible_cities_units_processed.csv"
OUTPUT_JSON = OUTPUT_DIR / "invisible_cities_units_processed.json"

In [3]:
if not RAW_TEXT_PATH.exists():
    raise FileNotFoundError(f"Text file not found: {RAW_TEXT_PATH}")

with open(RAW_TEXT_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Text length:", len(raw_text))

Text length: 148261


In [4]:
# Split by headings like "Cities & Memory 1"
pattern = r"(Cities.*?\d+)"

parts = re.split(pattern, raw_text)

units = []

for i in range(1, len(parts), 2):
    title = parts[i].strip()
    content = parts[i + 1].strip()

    units.append({
        "unit_id": len(units) + 1,
        "city_group": title,
        "text": content
    })

unit_df = pd.DataFrame(units)

print("Total units:", len(unit_df))
unit_df.head()

Total units: 46


,unit_id,city_group,text
0,1,Cities & Memory 1,]\n\nLeaving there and proceeding for three da...
1,2,Cities & Memory 2,]\n\nWhen a man rides a long time through wild...
2,3,Cities & Desire 1,]\n\nThere are two ways of describing the city...
3,4,Cities & Memory 3,"]\n\nIn vain, great-hearted Kublai, shall I at..."
4,5,Cities & Desire 2,"]\n\nAt the end of three days, moving southwar..."


In [5]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text.strip()

unit_df["clean_text"] = unit_df["text"].apply(clean_text)

unit_df.head()

,unit_id,city_group,text,clean_text
0,1,Cities & Memory 1,]\n\nLeaving there and proceeding for three da...,leaving there and proceeding for three days to...
1,2,Cities & Memory 2,]\n\nWhen a man rides a long time through wild...,when a man rides a long time through wild regi...
2,3,Cities & Desire 1,]\n\nThere are two ways of describing the city...,there are two ways of describing the city of d...
3,4,Cities & Memory 3,"]\n\nIn vain, great-hearted Kublai, shall I at...",in vain greathearted kublai shall i attempt to...
4,5,Cities & Desire 2,"]\n\nAt the end of three days, moving southwar...",at the end of three days moving southward you ...


In [6]:
from collections import Counter

def extract_keywords(text, top_k=10):
    words = text.split()
    common = Counter(words).most_common(top_k)
    return [w for w, _ in common]

unit_df["keywords"] = unit_df["clean_text"].apply(extract_keywords)

unit_df.head()

,unit_id,city_group,text,clean_text,keywords
0,1,Cities & Memory 1,]\n\nLeaving there and proceeding for three da...,leaving there and proceeding for three days to...,"[the, a, and, who, of, all, that, there, for, ..."
1,2,Cities & Memory 2,]\n\nWhen a man rides a long time through wild...,when a man rides a long time through wild regi...,"[the, a, he, city, where, isidora, with, is, i..."
2,3,Cities & Desire 1,]\n\nThere are two ways of describing the city...,there are two ways of describing the city of d...,"[the, and, in, that, of, you, city, can, each, i]"
3,4,Cities & Memory 3,"]\n\nIn vain, great-hearted Kublai, shall I at...",in vain greathearted kublai shall i attempt to...,"[the, of, and, a, in, city, as, it, i, like]"
4,5,Cities & Desire 2,"]\n\nAt the end of three days, moving southwar...",at the end of three days moving southward you ...,"[the, you, and, of, a, anastasia, to, with, i,..."


In [ ]:
def classify_city_type(keywords):
    keywords = set(keywords)

    if keywords & {"tower", "high", "height", "sky"}:
        return "vertical_city"

    if keywords & {"sand", "desert", "dry"}:
        return "desert_city"

    if keywords & {"ruin", "broken", "decay", "abandoned"}:
        return "ruin_city"

    if keywords & {"circle", "ring", "round"}:
        return "circular_city"

    if keywords & {"water", "floating", "sea"}:
        return "floating_city"

    return "surreal_city"

unit_df["city_type"] = unit_df["keywords"].apply(classify_city_type)

unit_df.head()

,unit_id,city_group,text,clean_text,keywords,city_type
0,1,Cities & Memory 1,]\n\nLeaving there and proceeding for three da...,leaving there and proceeding for three days to...,"[the, a, and, who, of, all, that, there, for, ...",surreal_city
1,2,Cities & Memory 2,]\n\nWhen a man rides a long time through wild...,when a man rides a long time through wild regi...,"[the, a, he, city, where, isidora, with, is, i...",surreal_city
2,3,Cities & Desire 1,]\n\nThere are two ways of describing the city...,there are two ways of describing the city of d...,"[the, and, in, that, of, you, city, can, each, i]",surreal_city
3,4,Cities & Memory 3,"]\n\nIn vain, great-hearted Kublai, shall I at...",in vain greathearted kublai shall i attempt to...,"[the, of, and, a, in, city, as, it, i, like]",surreal_city
4,5,Cities & Desire 2,"]\n\nAt the end of three days, moving southwar...",at the end of three days moving southward you ...,"[the, you, and, of, a, anastasia, to, with, i,...",surreal_city


In [8]:
unit_df.to_csv(OUTPUT_CSV, index=False)

unit_df.to_json(OUTPUT_JSON, orient="records", indent=2)

print("Saved to:")
print(OUTPUT_CSV)
print(OUTPUT_JSON)

Saved to:
d:\Work\Workspace\Projects\Python\data-driven-surface\data\processed\text\invisible_cities_units_processed.csv
d:\Work\Workspace\Projects\Python\data-driven-surface\data\processed\text\invisible_cities_units_processed.json


In [9]:
print("Total units:", len(unit_df))
print("\nCity type distribution:")
print(unit_df["city_type"].value_counts())

Total units: 46

City type distribution:
city_type
surreal_city    46
Name: count, dtype: int64
